# CS 483/583 · Module 14 · Reinforcement Learning and Sequential Decision-Making

A tiny delivery cart has two choices at a fork. It can finish immediately for two points or travel one more step for a larger delivery. Which route should it take if later rewards count less?

Module 13 generated samples by taking successive steps. Here a decision maker chooses at each step, and its choice changes which states and rewards it can reach. Follow one cart before writing a learning rule.

## Running the examples

Use a Python 3 notebook with PyTorch installed. A CPU is sufficient. Run the cells in order, then change an input and predict what will happen.

Two generators could produce different routes through a grid. What extra information would tell you which route to prefer?

<details><summary>Reveal answer</summary>

Specify what counts as success, the reward along each route and how long the cart can keep acting. A picture of a route alone gives no preference.

</details>

## Turn a fork into a decision problem

A state describes where the cart is now. An action is a choice available there. A transition tells us where the cart goes after the choice. A reward is a number the cart receives on that transition. Reaching Done ends the episode, one complete trip.

From Start, taking Short earns 2 and ends the trip. Taking Detour earns 0 and moves to Bridge. At Bridge the only action, Deliver, earns 4 and ends the trip. These are the complete rules of this first fictional grid.

**Formal notation.** Write a transition as (s, a, r, s′). Read this as current state s, chosen action a, received reward r, next state s′. The prime marks the next state.

**Predict before you run.** After Detour, can the cart take Short on its next move?

In [1]:
routes = {("Start", "Short"): (2, "Done"),
          ("Start", "Detour"): (0, "Bridge"),
          ("Bridge", "Deliver"): (4, "Done")}
state = "Start"
for action in ["Detour", "Deliver"]:
    reward, next_state = routes[(state, action)]
    print(state, action, reward, next_state)
    state = next_state

Start Detour 0 Bridge
Bridge Deliver 4 Done


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

It is now at Bridge. The only listed action there is Deliver. The earlier choice changed the choices available next.

</details>

## Decide what the state must remember

A Markov decision process (MDP) specifies states, actions, how likely each next state and reward are, and how future rewards count. The Markov assumption says the current state and action contain all the history needed for those probabilities.

Suppose a gate opens only for a charged cart. Two carts at the same gate could go different ways under the same action because their batteries differ. Include charge in the state before listing transition probabilities.

**Formal notation.** P(s′ | s, a) means the probability of a next state s′ given the present state s and action a. Probabilities across possible next states sum to 1.

**Predict before you run.** If a gate remembers the last code entered and that changes the next result, what could you add to the state?

In [2]:
next_state = {("Gate", True): "Across", ("Gate", False): "Charging bay"}
for charge in (True, False):
    state = ("Gate", charge)
    print(state, "attempt crossing", next_state[state])

('Gate', True) attempt crossing Across
('Gate', False) attempt crossing Charging bay


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Record the relevant code or whether the last code was valid. Location alone would mix histories with different consequences for the same action.

</details>

## Compare immediate and delayed reward

Add the rewards on a trip to get its return. A discount factor between zero and one shrinks a reward once for every step of delay. Count the first reward at its full value.

With discount 0.75, Short returns 2. Detour returns 0 + 0.75 × 4 = 3. Change the discount to 0.25 and Detour returns just 1. How we value waiting changes the best choice.

**Formal notation.** G = r₁ + γr₂ + γ²r₃ + … . G is the discounted return, γ (gamma) the discount, and r₁ the reward on the first transition.

**Predict before you run.** For what discount do Short and Detour tie?

In [3]:
for gamma in (0.25, 0.75):
    for route in ([2], [0, 4]):
        total = sum(gamma ** t * r for t, r in enumerate(route))
        print(gamma, route, total)

0.25 [2] 2.0
0.25 [0, 4] 1.0
0.75 [2] 2.0
0.75 [0, 4] 3.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Short returns 2 and Detour returns 4γ. Set 4γ = 2, giving γ = 0.5.

</details>

## Evaluate a policy before improving it

A policy tells the cart how to choose from the available actions. It can assign probabilities to choices. A state's value under a policy is the average return if trips start there and use that policy.

At γ = 0.75, suppose a policy takes Detour on half of trips and Short on half. The average starting return is 0.5 × 3 + 0.5 × 2 = 2.5.

**Formal notation.** π(a | s) is the policy's probability of choosing a at s. Vπ(s) = Eπ[G | s] is the mean return you would expect following π from s.

**Predict before you run.** What is the value at Bridge under any policy in this first grid?

In [4]:
gamma = 0.75
for detour_probability in (0.0, 0.5, 1.0):
    short_probability = 1 - detour_probability
    value = short_probability * 2 + detour_probability * gamma * 4
    print(detour_probability, value)

0.0 2.0
0.5 2.5
1.0 3.0


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

The cart must Deliver and receives 4 immediately. Done has no future reward, so Bridge is worth 4.

</details>

## Work backward with a Bellman backup

A backup updates a state's value by looking one transition ahead. For each available action, add its immediate reward to the discounted value of the next state. Choose the highest value if the goal is the best policy.

Start all values at zero and update all states from the previous round. Round one discovers that Bridge is worth 4, while Start sees only Short's 2. Round two carries Bridge's 4 back to Start and Detour wins with 3.

**Formal notation.** For a deterministic transition, Vnew(s) = maxₐ [r + γ Vold(s′)]. The subscript on max means compare available actions. At Done the remaining value is zero.

**Predict before you run.** Why did the first sweep fail to favor Detour?

In [5]:
import torch
v = torch.zeros(3)  # Start, Bridge, Done
for sweep in range(2):
    old = v.clone()
    v[0] = max(2.0, float(0.75 * old[1]))
    v[1] = 4.0
    print("sweep", sweep + 1, v.tolist())

sweep 1 [2.0, 4.0, 0.0]
sweep 2 [3.0, 4.0, 0.0]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

It used the old Bridge estimate of zero. Only the second synchronous sweep can use the four points discovered at Bridge in the first sweep.

</details>

## Average over an uncertain crossing

A known model sometimes lists several outcomes for one action. Weight each outcome's reward plus future value by its probability. Choose among actions after computing those averages.

Now change the grid. Detour reaches Bridge with probability 0.8 and ends empty-handed otherwise. The next delivery at Bridge still earns 4. At γ = 0.75 the expected Detour return is 0.8 × 3 + 0.2 × 0 = 2.4.

**Formal notation.** Q(s,a) = Σ p × (r + γV(s′)). Σ means sum over possible outcomes. Q is the expected return of choosing a now with the stated continuation values.

**Predict before you run.** If the crossing succeeds half the time, which route has higher expected return?

In [6]:
gamma = 0.75
for crossing in (0.5, 0.8):
    outcomes = [(crossing, 0, 4), (1-crossing, 0, 0)]
    q = sum(p * (reward + gamma * v_next)
            for p, reward, v_next in outcomes)
    print(round(q, 2), "Detour" if q > 2 else "Short")

1.5 Short
2.4 Detour


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

Detour has expected return 0.5 × 0.75 × 4 = 1.5. Short has return 2, so Short wins.

</details>

## Learn an action value from one trip

When the transition probabilities are unknown, the cart can update an action estimate from its observed reward and the best estimated action at the next state. The difference between that target and its old estimate tells it which way to move.

The estimate for Detour is 1. A successful transition earned 0 and the best next-action estimate is 4. With γ = 0.75 the target is 3. A learning rate of 0.5 moves the estimate halfway from 1 to 3, giving 2.

**Formal notation.** Qnew = Qold + α [r + γ max_b Q(s′,b) − Qold]. Here α (alpha) is the step size and b ranges over actions at the next state. Use 0 for the next value on a terminal transition.

**Predict before you run.** After the successful update, the next trip ends with zero reward on Detour. What changes?

In [7]:
import torch
q = torch.tensor([2., 1.])  # Short and Detour at Start
for next_value in (4., 0.):  # crossing succeeds, then fails on another trip
    target = 0. + 0.75 * next_value
    q[1] += 0.5 * (target - q[1])
    print(q.tolist())

[2.0, 2.0]
[2.0, 1.0]


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

That sample's target is zero. The previous estimate was 2, so the half-step update lowers it to 1. A single trip can differ from the average return.

</details>

## Explore a choice whose value is uncertain

Taking the action with the highest estimate is exploitation. Exploration tries other actions so their estimates can improve. In an epsilon-greedy policy with one best action, choose it directly with probability 1 − ε. Otherwise draw uniformly from all actions, including the current best.

If there are two actions and ε = 0.2, the best gets 0.8 + 0.1 = 0.9 probability. The other gets 0.1. During a separate evaluation of the learned greedy policy, set ε to zero and record fresh trip returns.

**Formal notation.** π(best | s) = 1 − ε + ε/k and π(other | s) = ε/k for k actions with a unique current best.

**Predict before you run.** Why could a cart that always chooses Short learn very little about Detour?

In [8]:
epsilon, count = 0.2, 2
for action, best in [("Short", True), ("Detour", False)]:
    p = epsilon / count + (1 - epsilon if best else 0.0)
    print(action, round(p, 2))

Short 0.9
Detour 0.1


Compare your reasoning with the worked answer.

<details><summary>Reveal answer</summary>

It never samples Detour's outcome from Start. Its estimate can remain at its initial value even when a better delayed return is available.

</details>

## Exercise 1

A corridor action earns −1 and moves to a room worth 8. Ending immediately earns 4. With γ = 0.75, which action wins?

<details><summary>Hint</summary>

Compute immediate reward plus discounted next value for each action.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The corridor is worth −1 + 0.75 × 8 = 5, versus 4 for ending. Choose the corridor.

</details>

## Exercise 2

A next state worth 10 occurs with probability 0.4. Otherwise the episode ends. The action always earns 1 and γ = 0.5. Back up its value.

<details><summary>Hint</summary>

Count the reward of 1 in both outcomes.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The two outcomes are worth 6 and 1. Their weighted mean is 0.4 × 6 + 0.6 × 1 = 3.

</details>

## Exercise 3

In a grid with locked doors and reusable keys, propose an MDP state and a terminal event.

<details><summary>Hint</summary>

Can two agents at the same location get different outcomes by trying the same door?

</details>

Check your work.

<details><summary>Reveal answer</summary>

Use location and key possession, plus door status if opening persists independently. Moving or unlocking is an action. Reaching the destination ends the episode. Specify a reward on each transition.

</details>

## Exercise 4

An action value is 5. A trip ends on the next move, earning 1. Use α = 0.25 to update it.

<details><summary>Hint</summary>

A terminal state has zero continuation value.

</details>

Check your work.

<details><summary>Reveal answer</summary>

The target is 1. Qnew = 5 + 0.25 × (1 − 5) = 4.

</details>

## Exercise 5

A cart has four actions and one current best. Compare ε = 0.4 during learning to ε = 0 in evaluation.

<details><summary>Hint</summary>

Uniform exploration can also draw the best action.

</details>

Check your work.

<details><summary>Reveal answer</summary>

During learning the best action gets 0.6 + 0.4/4 = 0.7, and each other action gets 0.1. With ε = 0 the cart chooses the current best every time. Evaluation on fresh trips measures this fixed policy.

</details>

Changing a crossing probability reversed the preferred route. Module 15 asks how to check whether a model's confidence and performance survive a change in the population where it will be used.

A simulator sets crossing success to 0.8. A field log shows 5 successes in 10 attempts. What would you check before using its recommendation?

<details><summary>Reveal answer</summary>

Check whether attempted crossings in the log represent future conditions, inspect more trials and compare observed frequencies with the assumed probability. The preferred route can change as the evidence changes.

</details>